In [5]:
import time
import tifffile
import matplotlib
import numpy as np

matplotlib.use("Agg")

import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

In [7]:
Z_raw = np.load("./data/fuji_elevation.npy").astype(np.float64)
lat   = np.load("./data/fuji_lat.npy")
lon   = np.load("./data/fuji_lon.npy")

# .npy хранят данные в порядке «север -Ю юг»,
# переворачиваем, чтобы y возрастал с юга на север
Z_raw  = Z_raw[::-1, :]
lat_sn = lat[::-1]

lat0 = 35.3606
deg2m_lat = 111_320.0
deg2m_lon = 111_320.0 * np.cos(np.radians(lat0))

x_m = (lon    - lon[0])    * deg2m_lon
y_m = (lat_sn - lat_sn[0]) * deg2m_lat

dx = x_m[1] - x_m[0]
dy = y_m[1] - y_m[0]
nrows, ncols = Z_raw.shape

print(f"\nСетка: {nrows}x{ncols}")
print(f"Шаг: dx = {dx:.1f} м, dy = {dy:.1f} м")
print(f"Область: {x_m[-1]/1000:.1f} x {y_m[-1]/1000:.1f} км")
print(f"Высота: {Z_raw.min():.0f} — {Z_raw.max():.0f} м")


Сетка: 1080x1080
Шаг: dx = 25.2 м, dy = 30.9 м
Область: 27.2 x 33.4 км
Высота: 63 — 3759 м


In [8]:

# ══════════════════════════════════════════════
# 2. Эталонный объём (2D трапеции)
# ══════════════════════════════════════════════

def volume_trapezoid_2d(Z, dx, dy, h0):
    """
    Объём над уровнем h0 по формуле 2D-трапеций на регулярной сетке.
    """
    H = np.maximum(Z - h0, 0)

    # Веса: углы=1/4, края=1/2, внутренние=1
    W = np.ones_like(H)
    W[0, :] *= 0.5;  W[-1, :] *= 0.5
    W[:, 0] *= 0.5;  W[:, -1] *= 0.5

    return dx * dy * np.sum(W * H)


h0 = 500.0  # базовый уровень (м)

V_ref = volume_trapezoid_2d(Z_raw, dx, dy, h0)
print(f"\n--- Эталонный объём (h₀ = {h0:.0f} м) ---")
print(f"V_ref = {V_ref:.6e} м³ = {V_ref/1e9:.3f} км³")




--- Эталонный объём (h₀ = 500 м) ---
V_ref = 5.134853e+11 м³ = 513.485 км³


In [9]:

# ══════════════════════════════════════════════
# 3. B-сплайновый базис (рекурсия Кокса–де Бора)
# ══════════════════════════════════════════════

def make_knot_vector(a, b, n_internal, degree=3):
    """Расширенный вектор узлов (clamped)."""
    internal = np.linspace(a, b, n_internal + 2)
    return np.concatenate([
        np.repeat(a, degree), internal, np.repeat(b, degree)
    ])


def bspline_basis(x, knots, degree=3):
    """Матрица B-сплайнового базиса (N × K)."""
    x = np.asarray(x, dtype=float)
    knots = np.asarray(knots, dtype=float)
    N = len(x)

    B = np.zeros((N, len(knots) - 1))
    for i in range(len(knots) - 1):
        if knots[i] < knots[i + 1]:
            B[(x >= knots[i]) & (x < knots[i + 1]), i] = 1.0
    for i in range(len(knots) - 2, -1, -1):
        if knots[i] < knots[i + 1]:
            B[x == knots[-1], i] = 1.0
            break

    for d in range(1, degree + 1):
        B_new = np.zeros((N, len(knots) - 1 - d))
        for i in range(len(knots) - 1 - d):
            d1 = knots[i + d] - knots[i]
            d2 = knots[i + d + 1] - knots[i + 1]
            if d1 > 0:
                B_new[:, i] += (x - knots[i]) / d1 * B[:, i]
            if d2 > 0:
                B_new[:, i] += (knots[i + d + 1] - x) / d2 * B[:, i + 1]
        B = B_new
    return B


def diff_matrix(K, order=2):
    """Матрица конечных разностей порядка order."""
    D = np.eye(K)
    for _ in range(order):
        D = D[1:, :] - D[:-1, :]
    return D


# ══════════════════════════════════════════════
# 4. 2D тензорный P-сплайн
# ══════════════════════════════════════════════

def pspline_2d(x, y, Z, n_basis_x=30, n_basis_y=30,
               degree=3, penalty_order=2, lam=1.0):
    """
    2D P-сплайн (тензорное произведение).

    Модель: Z ≈ Bx @ A @ By.T

    Векторизация:  vec(Z) ≈ (By ⊗ Bx) vec(A)

    Штраф:  λ_x ||Dx A||²_F + λ_y ||A Dy'||²_F

    Нормальные уравнения (Кронекер):
        [(By'By ⊗ Bx'Bx) + λ(I ⊗ Dx'Dx) + λ(Dy'Dy ⊗ I)] vec(A)
            = vec(Bx' Z By)
    """
    # Базисные матрицы
    kx = make_knot_vector(x[0], x[-1], n_basis_x, degree)
    ky = make_knot_vector(y[0], y[-1], n_basis_y, degree)
    Bx = bspline_basis(x, kx, degree)  # (nx, Kx)
    By = bspline_basis(y, ky, degree)  # (ny, Ky)

    Kx, Ky = Bx.shape[1], By.shape[1]

    # Матрицы разностей
    Dx = diff_matrix(Kx, penalty_order)
    Dy = diff_matrix(Ky, penalty_order)

    # Компоненты нормальных уравнений
    BxTBx = Bx.T @ Bx          # (Kx, Kx)
    ByTBy = By.T @ By          # (Ky, Ky)
    DxTDx = Dx.T @ Dx          # (Kx, Kx)
    DyTDy = Dy.T @ Dy          # (Ky, Ky)

    # Правая часть: vec(Bx' Z By)
    rhs_mat = Bx.T @ Z @ By    # (Kx, Ky)
    rhs = rhs_mat.ravel()       # (Kx*Ky,)

    # Левая часть: Кронекер-произведения
    Ix = np.eye(Kx)
    Iy = np.eye(Ky)

    LHS = (np.kron(ByTBy, BxTBx)
           + lam * np.kron(Iy, DxTDx)
           + lam * np.kron(DyTDy, Ix))

    # Решаем систему
    alpha_vec = np.linalg.solve(LHS, rhs)
    A = alpha_vec.reshape(Kx, Ky)

    return A, Bx, By, kx, ky


def evaluate_2d_pspline(A, kx, ky, x_eval, y_eval, degree=3):
    """Вычисление P-сплайна на сетке (x_eval, y_eval)."""
    Bx = bspline_basis(x_eval, kx, degree)
    By = bspline_basis(y_eval, ky, degree)
    return Bx @ A @ By.T



In [10]:

# ══════════════════════════════════════════════
# 5. Подгонка и сравнение для разных параметров
# ══════════════════════════════════════════════

print("\n" + "=" * 65)
print("ПОДГОНКА 2D P-СПЛАЙНА")
print("=" * 65)

configs = [
    (15, 1.0),
    (25, 1.0),
    (25, 0.01),
    (40, 1.0),
    (40, 0.01),
    (60, 0.001),
]

results = []

for n_basis, lam in configs:
    label = f"K={n_basis}, λ={lam}"
    print(f"\n--- {label} ---")

    t0 = time.time()
    A, Bx, By, kx, ky = pspline_2d(
        x_m, y_m, Z_raw,
        n_basis_x=n_basis, n_basis_y=n_basis,
        degree=3, penalty_order=2, lam=lam
    )
    t_fit = time.time() - t0

    # Восстановим поверхность на исходной сетке
    t0 = time.time()
    Z_ps = evaluate_2d_pspline(A, kx, ky, x_m, y_m, degree=3)
    t_eval = time.time() - t0

    # Объём P-сплайна
    V_ps = volume_trapezoid_2d(Z_ps, dx, dy, h0)

    # Ошибки
    err_max = np.max(np.abs(Z_ps - Z_raw))
    err_rmse = np.sqrt(np.mean((Z_ps - Z_raw) ** 2))
    rel_vol = abs(V_ps - V_ref) / V_ref * 100

    Kx = Bx.shape[1]
    Ky = By.shape[1]
    print(f"  Базис: {Kx}×{Ky} = {Kx*Ky} коэфф.")
    print(f"  Подгонка: {t_fit:.2f} с, вычисление: {t_eval:.2f} с")
    print(f"  Max err = {err_max:.1f} м, RMSE = {err_rmse:.2f} м")
    print(f"  V_pspline = {V_ps:.6e} м³ = {V_ps/1e9:.3f} км³")
    print(f"  Ошибка объёма: {rel_vol:.4f}%")

    results.append({
        'label': label, 'n_basis': n_basis, 'lam': lam,
        'Kx': Kx, 'Ky': Ky,
        'err_max': err_max, 'err_rmse': err_rmse,
        'V_ps': V_ps, 'rel_vol': rel_vol,
        'Z_ps': Z_ps, 'A': A, 'kx': kx, 'ky': ky,
    })




ПОДГОНКА 2D P-СПЛАЙНА

--- K=15, λ=1.0 ---
  Базис: 19×19 = 361 коэфф.
  Подгонка: 0.05 с, вычисление: 0.02 с
  Max err = 268.1 м, RMSE = 31.74 м
  V_pspline = 5.134579e+11 м³ = 513.458 км³
  Ошибка объёма: 0.0053%

--- K=25, λ=1.0 ---
  Базис: 29×29 = 841 коэфф.
  Подгонка: 0.14 с, вычисление: 0.02 с
  Max err = 279.7 м, RMSE = 23.73 м
  V_pspline = 5.134618e+11 м³ = 513.462 км³
  Ошибка объёма: 0.0046%

--- K=25, λ=0.01 ---
  Базис: 29×29 = 841 коэфф.
  Подгонка: 0.06 с, вычисление: 0.01 с
  Max err = 289.1 м, RMSE = 23.07 м
  V_pspline = 5.134652e+11 м³ = 513.465 км³
  Ошибка объёма: 0.0039%

--- K=40, λ=1.0 ---
  Базис: 44×44 = 1936 коэфф.
  Подгонка: 0.31 с, вычисление: 0.01 с
  Max err = 188.8 м, RMSE = 17.50 м
  V_pspline = 5.134625e+11 м³ = 513.462 км³
  Ошибка объёма: 0.0044%

--- K=40, λ=0.01 ---
  Базис: 44×44 = 1936 коэфф.
  Подгонка: 0.27 с, вычисление: 0.01 с
  Max err = 188.7 м, RMSE = 16.46 м
  V_pspline = 5.134682e+11 м³ = 513.468 км³
  Ошибка объёма: 0.0033%

--- K=6

In [11]:

# ══════════════════════════════════════════════
# 6. Итоговая таблица
# ══════════════════════════════════════════════

print("\n" + "=" * 80)
print("ИТОГОВАЯ СВОДКА")
print("=" * 80)
print(f"Эталонный объём (h₀={h0:.0f} м): V_ref = {V_ref/1e9:.4f} км³")
print()
print(f"{'Конфигурация':<18} {'Коэфф':>7} {'Max err,м':>10} {'RMSE,м':>9} "
      f"{'V, км³':>10} {'δV, %':>8}")
print("-" * 80)
for r in results:
    print(f"{r['label']:<18} {r['Kx']*r['Ky']:>7d} {r['err_max']:>10.1f} "
          f"{r['err_rmse']:>9.2f} {r['V_ps']/1e9:>10.4f} {r['rel_vol']:>8.4f}")




ИТОГОВАЯ СВОДКА
Эталонный объём (h₀=500 м): V_ref = 513.4853 км³

Конфигурация         Коэфф  Max err,м    RMSE,м     V, км³    δV, %
--------------------------------------------------------------------------------
K=15, λ=1.0            361      268.1     31.74   513.4579   0.0053
K=25, λ=1.0            841      279.7     23.73   513.4618   0.0046
K=25, λ=0.01           841      289.1     23.07   513.4652   0.0039
K=40, λ=1.0           1936      188.8     17.50   513.4625   0.0044
K=40, λ=0.01          1936      188.7     16.46   513.4682   0.0033
K=60, λ=0.001         4096      148.5     11.80   513.4715   0.0027


In [12]:

# ══════════════════════════════════════════════
# 7. Графики
# ══════════════════════════════════════════════

cmap_terrain = LinearSegmentedColormap.from_list('fuji', [
    (0.00, '#1a472a'), (0.08, '#2d6a3e'), (0.15, '#4a8c3f'),
    (0.25, '#7a9a4a'), (0.35, '#a08050'), (0.50, '#8b6942'),
    (0.65, '#a0896a'), (0.80, '#b8a898'), (0.90, '#d0d0d0'),
    (1.00, '#ffffff'),
])

# Выберем лучший результат
best = min(results, key=lambda r: r['err_rmse'])
Z_best = best['Z_ps']

fig, axes = plt.subplots(2, 3, figsize=(20, 13))
fig.suptitle(f"2D P-сплайн для горы Фудзи  |  Эталонный объём = {V_ref/1e9:.3f} км³ "
             f"(h₀ = {h0:.0f} м)", fontsize=14, fontweight='bold')

X_km = x_m / 1000
Y_km = y_m / 1000

# 1) Исходные данные
ax = axes[0, 0]
im = ax.imshow(Z_raw, extent=[X_km[0], X_km[-1], Y_km[0], Y_km[-1]],
               origin='lower', cmap=cmap_terrain, vmin=0, vmax=3800)
ax.set_title("Исходные SRTM-данные", fontsize=11)
ax.set_xlabel('Восток (км)'); ax.set_ylabel('Север (км)')
plt.colorbar(im, ax=ax, label='м', shrink=0.85)

# 2) P-сплайн (лучший)
ax = axes[0, 1]
im = ax.imshow(Z_best, extent=[X_km[0], X_km[-1], Y_km[0], Y_km[-1]],
               origin='lower', cmap=cmap_terrain, vmin=0, vmax=3800)
ax.set_title(f"P-сплайн ({best['label']})", fontsize=11)
ax.set_xlabel('Восток (км)'); ax.set_ylabel('Север (км)')
plt.colorbar(im, ax=ax, label='м', shrink=0.85)

# 3) Ошибка
ax = axes[0, 2]
err = Z_best - Z_raw
im = ax.imshow(err, extent=[X_km[0], X_km[-1], Y_km[0], Y_km[-1]],
               origin='lower', cmap='RdBu_r', vmin=-150, vmax=150)
ax.set_title(f"Ошибка (max={best['err_max']:.0f} м, RMSE={best['err_rmse']:.1f} м)",
             fontsize=11)
ax.set_xlabel('Восток (км)'); ax.set_ylabel('Север (км)')
plt.colorbar(im, ax=ax, label='м', shrink=0.85)

# 4) Сечение через вершину (запад–восток)
ax = axes[1, 0]
peak_row = np.argmax(np.max(Z_raw, axis=1))
ax.plot(X_km, Z_raw[peak_row, :], 'k-', lw=1.5, label='SRTM', alpha=0.6)
for r in results:
    ax.plot(X_km, r['Z_ps'][peak_row, :], '--', lw=1.2,
            label=f"{r['label']} (δV={r['rel_vol']:.3f}%)")
ax.axhline(h0, color='gray', ls=':', lw=1, label=f'h₀ = {h0:.0f} м')
ax.set_title(f"Сечение запад–восток через вершину (строка {peak_row})", fontsize=11)
ax.set_xlabel('Восток (км)'); ax.set_ylabel('Высота (м)')
ax.legend(fontsize=7, loc='upper right')
ax.grid(True, alpha=0.3)

# 5) Сечение через вершину (юг–север)
ax = axes[1, 1]
peak_col = np.argmax(np.max(Z_raw, axis=0))
ax.plot(Y_km, Z_raw[:, peak_col], 'k-', lw=1.5, label='SRTM', alpha=0.6)
for r in results:
    ax.plot(Y_km, r['Z_ps'][:, peak_col], '--', lw=1.2,
            label=f"{r['label']} (δV={r['rel_vol']:.3f}%)")
ax.axhline(h0, color='gray', ls=':', lw=1, label=f'h₀ = {h0:.0f} м')
ax.set_title(f"Сечение юг–север через вершину (столбец {peak_col})", fontsize=11)
ax.set_xlabel('Север (км)'); ax.set_ylabel('Высота (м)')
ax.legend(fontsize=7, loc='upper right')
ax.grid(True, alpha=0.3)

# 6) Ошибка объёма vs число коэффициентов
ax = axes[1, 2]
n_coeffs = [r['Kx'] * r['Ky'] for r in results]
vol_errs = [r['rel_vol'] for r in results]
rmses = [r['err_rmse'] for r in results]
labels = [r['label'] for r in results]

ax.bar(range(len(results)), vol_errs, color='steelblue', alpha=0.8)
ax.set_xticks(range(len(results)))
ax.set_xticklabels(labels, rotation=25, fontsize=8)
ax.set_ylabel('Ошибка объёма δV (%)', fontsize=10)
ax.set_title("Ошибка объёма по конфигурациям", fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

# Добавляем RMSE как текст
for i, (ve, rm) in enumerate(zip(vol_errs, rmses)):
    ax.text(i, ve + 0.01, f"RMSE\n{rm:.1f} м", ha='center', fontsize=7)

plt.tight_layout()
plt.savefig("/mnt/user-data/outputs/fuji_pspline_volume.png",
            dpi=180, bbox_inches='tight')
plt.close()
print("\n✓ fuji_pspline_volume.png")



FileNotFoundError: [Errno 2] No such file or directory: '/mnt/user-data/outputs/fuji_pspline_volume.png'

In [ ]:

# ══════════════════════════════════════════════
# 8. 3D-сравнение: оригинал vs P-сплайн
# ══════════════════════════════════════════════

step = 5
Xs, Ys = np.meshgrid(X_km[::step], Y_km[::step])

fig2 = plt.figure(figsize=(20, 9))

ax1 = fig2.add_subplot(121, projection='3d')
ax1.plot_surface(Xs, Ys, Z_raw[::step, ::step], cmap=cmap_terrain,
                 rstride=1, cstride=1, linewidth=0, antialiased=True,
                 vmin=0, vmax=3800)
ax1.set_title("SRTM (оригинал)", fontsize=13, fontweight='bold')
ax1.set_xlabel('Восток (км)'); ax1.set_ylabel('Север (км)')
ax1.set_zlabel('м')
ax1.view_init(elev=30, azim=225)
ax1.set_box_aspect([1, 1, 0.45])

ax2 = fig2.add_subplot(122, projection='3d')
ax2.plot_surface(Xs, Ys, Z_best[::step, ::step], cmap=cmap_terrain,
                 rstride=1, cstride=1, linewidth=0, antialiased=True,
                 vmin=0, vmax=3800)
ax2.set_title(f"P-сплайн ({best['label']})\nδV = {best['rel_vol']:.4f}%",
              fontsize=13, fontweight='bold')
ax2.set_xlabel('Восток (км)'); ax2.set_ylabel('Север (км)')
ax2.set_zlabel('м')
ax2.view_init(elev=30, azim=225)
ax2.set_box_aspect([1, 1, 0.45])

plt.tight_layout()
plt.savefig("/mnt/user-data/outputs/fuji_3d_comparison.png",
            dpi=180, bbox_inches='tight')
plt.close()
print("✓ fuji_3d_comparison.png")